In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")

Unrecognized keys in `rope_scaling` for 'rope_type'='linear': {'type'}


In [18]:
from torch import nn
for name,module in model.named_modules():
    if isinstance(module,nn.Linear):
        print("******",name,"******")
        for name,param in module.named_parameters():
            a = param.flatten()
            print(a.shape)

****** model.layers.0.self_attn.q_proj ******
torch.Size([4194304])
****** model.layers.0.self_attn.k_proj ******
torch.Size([4194304])
****** model.layers.0.self_attn.v_proj ******
torch.Size([4194304])
****** model.layers.0.self_attn.o_proj ******
torch.Size([4194304])
****** model.layers.0.mlp.gate_proj ******
torch.Size([11272192])
****** model.layers.0.mlp.up_proj ******
torch.Size([11272192])
****** model.layers.0.mlp.down_proj ******
torch.Size([11272192])
****** model.layers.1.self_attn.q_proj ******
torch.Size([4194304])
****** model.layers.1.self_attn.k_proj ******
torch.Size([4194304])
****** model.layers.1.self_attn.v_proj ******
torch.Size([4194304])
****** model.layers.1.self_attn.o_proj ******
torch.Size([4194304])
****** model.layers.1.mlp.gate_proj ******
torch.Size([11272192])
****** model.layers.1.mlp.up_proj ******
torch.Size([11272192])
****** model.layers.1.mlp.down_proj ******
torch.Size([11272192])
****** model.layers.2.self_attn.q_proj ******
torch.Size([419430

In [26]:
import torch 
a = torch.randn(4,5)
print(a,"a")
print(a.shape,"a shape")
max_a = a.max(1,keepdim=True)
print(max_a.values,"max a")
x,y = max_a.values.shape
print(x,y)

tensor([[-0.4105,  0.2742,  0.9076,  0.5351,  1.4412],
        [-1.4949, -0.6830, -1.1749,  0.0687, -1.4974],
        [-0.0807,  1.2598, -0.1753,  0.7882, -0.2182],
        [-0.1103,  0.8900, -0.8188, -0.5733, -0.5329]]) a
torch.Size([4, 5]) a shape
tensor([[1.4412],
        [0.0687],
        [1.2598],
        [0.8900]]) max a
4 1


In [ ]:
from torch import nn
import torch.nn.functional as F
class int8Linear(nn.Module):
    def __init__(self,model):
        super().__init__()
        # should take only nn.Linear modules
        self.model = model
        self.quantized_weights = None
        self.bias = model.bias
        self.scale_values = None
        self.quantize()
    
    def forward(self,input):
        weights_fp16 = self.dequantize()
        return F.linear(input,weights_fp16,self.bias)

    def dequantize(self):

        original_shape = self.quantized_weights.shape
        original_numel = self.quantized_weights.numel()
        flat = self.quantized_weights.flatten()
        remainder = flat.numel()%64
        if remainder:
            flat = F.pad(flat,(0,64-remainder))
        blocks = flat.view(-1,64)
        blocks = blocks.to(torch.float16)*self.scale_values
        param = blocks.flatten()[:original_numel].view(original_shape)
        return param
    
    def quantize(self):
        # Let's quantize every tensor 
        param = self.model.weight.data
        # first we to flatten 
        original_shape = param.shape
        original_numel = param.numel()
        flat = param.flatten()
        # now divide into blocks 
        remainder = flat.numel()%64
        if remainder:
            flat = F.pad(flat,(0,64-remainder))
        blocks = flat.view(-1,64)
        # find the max values per block 
        max_values = blocks.abs().max(1,keepdim=True).values
        # calculate scale
        scale_values = max_values/127
        # compress teh data 
        blocks = blocks/scale_values
        # clamp and convert to int 8
        blocks = blocks.clamp(-128,127).to(torch.int8)
        # revert back to actual param shape 
        param = blocks.flatten()[:original_numel].view(original_shape)
        # setting the weights and bias 
        self.quantized_weights = param
        self.scale_values = scale_values
        return             
                    

                
